In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
import numpy as np
import google.generativeai as genai

In [ ]:
pdf_path = "/home/nineleaps/Downloads/Mini-RAG-chatbot-main/sample_data/SQL_Lecture_Notes.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text()

print("PDF Loaded Successfully")

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(text)

print("Number of Chunks:", len(chunks))

In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Vectors Stored:", index.ntotal)

In [ ]:
def retrieve_context(query, top_k=3):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    contexts = []

    for idx in indices[0]:
        contexts.append(chunks[idx])

    return contexts

In [ ]:
from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")


genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

 # Only first few chars

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content("Hello")

print(response.text)

In [ ]:
def generate_answer(question):

    contexts = retrieve_context(question, top_k=10)

    print("Retrieved Chunks:")
    print(contexts)

    context_text = "\n\n".join(contexts)

    prompt = f"""
You are a helpful assistant.

Answer ONLY using the provided context.

Context:
{context_text}

Question:
{question}
"""

    response = model.generate_content(prompt)

    print("Gemini Response Object:")
    print(response)

    return response.text

In [ ]:
while True:

    question = input("\nAsk a question: ")

    if question.lower() == "exit":
        break

    answer = generate_answer(question)

    print("\nAnswer:")
    print(answer)